
## Imports


In [20]:
# Install the pandera library using pip. This library is used for data validation.
!{sys.executable} -m pip install pandera

In [19]:
# Import necessary libraries for data manipulation, validation and file paths
import pandas as pd
import pandera.pandas as pa
from pandera import Column, Check
from pathlib import Path
import sys

# Add parent directory to path to allow importing local modules.
sys.path.append("..")

# Import data/model directory paths from project config.
from production.config import DATA_DIR

# Load health/fitness dataset.
df = pd.read_csv(DATA_DIR / "health_fitness.csv")

## Initial Inspection

The dataset contains 687,701 records after removing non-essential columns.
Most values are within valid ranges, though negative daily_steps and low calories_burned need correction.


In [22]:
# Load the dataset into DataFrame 'df'.
df = pd.read_csv(DATA_DIR / "health_fitness.csv")

In [23]:
# Preview Data Frame
df.head()

,participant_id,date,age,gender,height_cm,weight_kg,activity_type,duration_minutes,intensity,calories_burned,...,stress_level,daily_steps,hydration_level,bmi,resting_heart_rate,blood_pressure_systolic,blood_pressure_diastolic,health_condition,smoking_status,fitness_level
0,1,2024-01-01,56,F,165.3,53.7,Dancing,41,Low,3.3,...,3,7128,1.5,19.6,69.5,110.7,72.9,NaN,Never,0.04
1,1,2024-01-04,56,F,165.3,53.9,Swimming,28,Low,2.9,...,7,7925,1.8,19.6,69.5,110.7,72.9,NaN,Never,0.07
2,1,2024-01-05,56,F,165.3,54.2,Swimming,21,Medium,2.6,...,7,7557,2.7,19.6,69.5,110.7,72.9,NaN,Never,0.09
3,1,2024-01-07,56,F,165.3,54.4,Weight Training,99,Medium,10.7,...,8,11120,2.6,19.6,69.5,110.7,72.9,NaN,Never,0.21
4,1,2024-01-09,56,F,165.3,54.7,Swimming,100,Medium,12.7,...,1,5406,1.5,19.6,69.5,110.7,72.9,NaN,Never,0.33


In [24]:
# Data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 687701 entries, 0 to 687700
Data columns (total 22 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   participant_id            687701 non-null  int64  
 1   date                      687701 non-null  object 
 2   age                       687701 non-null  int64  
 3   gender                    687701 non-null  object 
 4   height_cm                 687701 non-null  float64
 5   weight_kg                 687701 non-null  float64
 6   activity_type             687701 non-null  object 
 7   duration_minutes          687701 non-null  int64  
 8   intensity                 687701 non-null  object 
 9   calories_burned           687701 non-null  float64
 10  avg_heart_rate            687701 non-null  int64  
 11  hours_sleep               687701 non-null  float64
 12  stress_level              687701 non-null  int64  
 13  daily_steps               687701 non-null  i

In [25]:
# Overview of columns
df.columns

Index(['participant_id', 'date', 'age', 'gender', 'height_cm', 'weight_kg',
       'activity_type', 'duration_minutes', 'intensity', 'calories_burned',
       'avg_heart_rate', 'hours_sleep', 'stress_level', 'daily_steps',
       'hydration_level', 'bmi', 'resting_heart_rate',
       'blood_pressure_systolic', 'blood_pressure_diastolic',
       'health_condition', 'smoking_status', 'fitness_level'],
      dtype='object')

In [26]:
print(f"Shape before cleaning: {df.shape}")

Shape before cleaning: (687701, 22)


-------
## Data Quality Checks
--------------

- Duplicates
- Missing Values
- Pandera: data engineering validations (known ranges, data types, expected values)[1][2][3]
----------
#### **References**
--------
1. Applause, “Training Data, Validation Data and Test Data in Machine Learning (ML),” Applause Blog, Jan. 15, 2025. Available: https://www.applause.com/blog/training-data-validation-data-vs-test-data/

2. Pandera Documentation, “pandera.api.checks.Check,” Pandera 0.25.0 API Reference, Aug. 2025. Available: https://pandera.readthedocs.io/en/stable/reference/generated/pandera.api.checks.Check.html 

3. A. Tomassini, “Cultivating Data Integrity in Data Science with Pandera – Advanced validation techniques with Pandera to promote data quality and reliability,” Medium / TDS Archive, Dec. 22 2023. Available: https://medium.com/data-science/cultivating-data-integrity-in-data-science-with-pandera-2289608626cc

In [27]:
# 1. Checks for duplicated rows
duplicates = df.duplicated().sum()
print("Total duplicate rows:", duplicates)

Total duplicate rows: 0


In [28]:
# 2. Check for missing values
df.isna().sum().sort_values(ascending=False)

health_condition            490275
participant_id                   0
date                             0
smoking_status                   0
blood_pressure_diastolic         0
blood_pressure_systolic          0
resting_heart_rate               0
bmi                              0
hydration_level                  0
daily_steps                      0
stress_level                     0
hours_sleep                      0
avg_heart_rate                   0
calories_burned                  0
intensity                        0
duration_minutes                 0
activity_type                    0
weight_kg                        0
height_cm                        0
gender                           0
age                              0
fitness_level                    0
dtype: int64

In [29]:
# Dropping unwanted columns
drop_cols = ['participant_id', 'date', 'avg_heart_rate', 'bmi', 
    'resting_heart_rate','blood_pressure_systolic', 'blood_pressure_diastolic',
    'health_condition', 'smoking_status', 'fitness_level']

df = df.drop(columns=drop_cols)

In [30]:
# Checking drop cols worked
df.head()

,age,gender,height_cm,weight_kg,activity_type,duration_minutes,intensity,calories_burned,hours_sleep,stress_level,daily_steps,hydration_level
0,56,F,165.3,53.7,Dancing,41,Low,3.3,6.6,3,7128,1.5
1,56,F,165.3,53.9,Swimming,28,Low,2.9,8.1,7,7925,1.8
2,56,F,165.3,54.2,Swimming,21,Medium,2.6,6.2,7,7557,2.7
3,56,F,165.3,54.4,Weight Training,99,Medium,10.7,7.2,8,11120,2.6
4,56,F,165.3,54.7,Swimming,100,Medium,12.7,7.1,1,5406,1.5


### PANDERA - Data Validation

In [31]:
# Define a pandera DataFrame schema with data types and validation checks for each column.
schema = pa.DataFrameSchema({
    'age': Column(int),
    'gender': Column('str', Check.isin(['F', 'M', 'Other'])),
    'height_cm': Column('float', Check.in_range(145.0, 200.0)),
    'weight_kg': Column('float', Check.in_range(40, 200)),
    'activity_type': Column('str', Check.isin(['Dancing','Swimming','Weight Training',
                                                'HIIT','Running','Walking','Tennis', 
                                                'Basketball', 'Yoga', 'Cycling'])),
    'duration_minutes': Column('int', Check.in_range(5, 120)),
    'intensity': Column('str', Check.isin(['Low','Medium','High'])),
    'hours_sleep': Column('float', Check.in_range(1,12)),
    'stress_level': Column('int', Check.in_range(1,10)),
    'daily_steps': Column('int', [Check.gt(0), Check.le(40000)]),
    'hydration_level': Column('float', [Check.ge(0), Check.lt(6)]),                      
                                    
})

In [32]:
# Attempt to validate the DataFrame against the defined schema.
try:
    validated_df = schema.validate(df, lazy=True)
except pa.errors.SchemaErrors as e:
    print('Validation failed')
    display(e.failure_cases)

Validation failed


,schema_context,column,check,check_number,failure_case,index
0,Column,daily_steps,greater_than(0),0,-419,207781
1,Column,daily_steps,greater_than(0),0,-81,360838


In [33]:
# Drop negative steps safely as only 2
df = df[df['daily_steps'] >= 0]

In [34]:
# Display the 'daily_steps' column to verify the removal of negative values.
df['daily_steps']

0          7128
1          7925
2          7557
3         11120
4          5406
          ...  
687696     6911
687697     8932
687698     8864
687699     7455
687700     9737
Name: daily_steps, Length: 687699, dtype: int64

In [35]:
# Visualise top 5 value count, min and max
for col in df.columns:
    print("\n", df[col].value_counts().head(5))
    print(f"Min: {df[col].min()}")
    print(f"Max: {df[col].max()}")


 age
24    18125
44    18107
61    17806
60    17211
63    17153
Name: count, dtype: int64
Min: 18
Max: 64

 gender
F        338855
M        334023
Other     14821
Name: count, dtype: int64
Min: F
Max: Other

 height_cm
162.0    4856
163.5    4333
165.5    4080
168.7    3950
168.4    3943
Name: count, dtype: int64
Min: 145.0
Max: 198.5

 weight_kg
98.7    1208
96.4    1206
96.9    1200
99.5    1197
98.2    1197
Name: count, dtype: int64
Min: 45.3
Max: 188.4

 activity_type
Yoga               69961
Weight Training    69661
HIIT               69376
Dancing            69193
Cycling            69187
Name: count, dtype: int64
Min: Basketball
Max: Yoga

 duration_minutes
104    7018
22     6946
52     6939
58     6934
63     6927
Name: count, dtype: int64
Min: 20
Max: 120

 intensity
Medium    343433
Low       206910
High      137356
Name: count, dtype: int64
Min: High
Max: Medium

 calories_burned
6.4    3754
6.8    3737
6.1    3728
8.7    3723
7.9    3703
Name: count, dtype: int64
Min: 0.

### Save data 

In [36]:
# Define the output path for the cleaned DataFrame.
OUTPUT_PATH = Path("../data/latest.csv")   
# Save the cleaned DataFrame to a CSV file.
df.to_csv(OUTPUT_PATH, index=False)